# LunarSite — Stage 2: Terrain Segmentation (A100 GPU)

**Two runs:** ResNet-34 baseline, then DINOv2 enhanced. Compare results side by side.

Lunar augmentations: shadow rotation, extreme contrast, Hapke BRDF.

MC Dropout uncertainty maps on DINOv2 model.

**Runtime:** `Runtime > Change runtime type > A100 GPU`

In [ ]:
!pip install -q segmentation-models-pytorch albumentations kagglehub

In [ ]:
import gc
import json
import time
from pathlib import Path

import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
import psutil
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")

In [ ]:
dataset_path = Path(kagglehub.dataset_download("romainpessia/artificial-lunar-rocky-landscape-dataset"))
image_dir = dataset_path / "images" / "render"
mask_dir = dataset_path / "images" / "clean"
real_moon_dir = dataset_path / "real_moon_images"
print(f"Images: {len(list(image_dir.glob('render*.png')))}")

In [ ]:
COLOR_TO_CLASS = {(0,0,0): 0, (255,0,0): 1, (0,255,0): 2, (0,0,255): 3}
CLASS_COLORS = np.array([[0,0,0],[255,165,0],[255,0,0],[135,206,235]], dtype=np.uint8)
CLASS_NAMES = ["background", "small_rocks", "large_rocks", "sky"]
NC = 4
SEED = 42
INPUT_SIZE = 480

## Lunar Augmentations + Dataset

In [ ]:
class LunarShadowRotation(ImageOnlyTransform):
    def __init__(self, angle_range=(-45,45), always_apply=False, p=0.5):
        super().__init__(always_apply=always_apply, p=p)
        self.angle_range = angle_range
    def apply(self, img, angle=0, **p):
        gray = np.mean(img.astype(np.float32), axis=-1)
        sh = gray < 25
        if sh.sum() < 100: return img
        h,w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2,h//2), angle, 1.0)
        rot = cv2.warpAffine(sh.astype(np.uint8), M, (w,h), flags=cv2.INTER_NEAREST).astype(bool)
        r = img.copy()
        r[rot] = np.clip(r[rot]*0.1, 0, 255).astype(np.uint8)
        r[sh & ~rot] = np.clip(r[sh & ~rot]*3.0+30, 0, 200).astype(np.uint8)
        return r
    def get_params(self): return {"angle": np.random.uniform(*self.angle_range)}
    def get_transform_init_args_names(self): return ("angle_range",)

class ExtremeContrast(ImageOnlyTransform):
    def __init__(self, always_apply=False, p=0.5): super().__init__(always_apply=always_apply, p=p)
    def apply(self, img, sf=0.05, bf=1.5, **p):
        g = np.mean(img.astype(np.float32), axis=-1); med = np.median(g)
        r = img.astype(np.float32); r[g<med*0.5]*=sf; r[g>med*1.5]*=bf
        return np.clip(r,0,255).astype(np.uint8)
    def get_params(self): return {"sf":np.random.uniform(0.01,0.15),"bf":np.random.uniform(1.2,2.5)}
    def get_transform_init_args_names(self): return ()

class HapkeBRDF(ImageOnlyTransform):
    def __init__(self, always_apply=False, p=0.4): super().__init__(always_apply=always_apply, p=p)
    def apply(self, img, af=1.0, pf=1.0, **p):
        r = img.astype(np.float32)*af; g = np.mean(r,axis=-1,keepdims=True)
        n = g/(g.max()+1e-8); r *= (1-n*(1-n)*4*(1-pf))
        return np.clip(r,0,255).astype(np.uint8)
    def get_params(self): return {"af":np.random.uniform(0.5,1.5),"pf":np.random.uniform(0.7,1.3)}
    def get_transform_init_args_names(self): return ()

def get_transforms(sz, train):
    if train:
        return A.Compose([
            A.RandomCrop(sz,sz), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=(-0.3,0.2), contrast_limit=(-0.2,0.4), p=0.4),
            A.GaussNoise(p=0.2),
            LunarShadowRotation(p=0.3), ExtremeContrast(p=0.3), HapkeBRDF(p=0.3),
        ])
    return A.Compose([A.CenterCrop(sz,sz)])

class LunarDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths, self.mask_paths, self.transform = image_paths, mask_paths, transform
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        msk = cv2.cvtColor(cv2.imread(str(self.mask_paths[idx])), cv2.COLOR_BGR2RGB)
        h,w = msk.shape[:2]; mask = np.zeros((h,w), dtype=np.int64)
        for color, cls in COLOR_TO_CLASS.items(): mask[np.all(msk==color, axis=-1)] = cls
        if self.transform:
            t = self.transform(image=img, mask=mask); img, mask = t["image"], t["mask"]
        return {"image": torch.from_numpy(img.transpose(2,0,1).astype(np.float32)/255.0),
                "mask": torch.from_numpy(mask.astype(np.int64))}

# Split
all_imgs = sorted(image_dir.glob("render*.png"))
all_masks = sorted(mask_dir.glob("clean*.png"))
n = len(all_imgs)
perm = np.random.RandomState(SEED).permutation(n).tolist()
nt, nv = int(n*0.8), int(n*0.1)
train_idx, val_idx, test_idx = perm[:nt], perm[nt:nt+nv], perm[nt+nv:]

train_set = LunarDataset([all_imgs[i] for i in train_idx], [all_masks[i] for i in train_idx], get_transforms(INPUT_SIZE, True))
val_set = LunarDataset([all_imgs[i] for i in val_idx], [all_masks[i] for i in val_idx], get_transforms(INPUT_SIZE, False))
test_set = LunarDataset([all_imgs[i] for i in test_idx], [all_masks[i] for i in test_idx], get_transforms(INPUT_SIZE, False))
print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

## Training & Eval Functions

In [ ]:
class DiceCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode="multiclass")
        self.ce = nn.CrossEntropyLoss()
    def forward(self, p, t): return 0.5*self.dice(p,t) + 0.5*self.ce(p,t)

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0.0
    for batch in loader:
        imgs, masks = batch["image"].to(device), batch["mask"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        total += loss.item() * imgs.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, nc):
    model.eval()
    total_loss = 0.0
    inter = torch.zeros(nc); uni = torch.zeros(nc)
    di = torch.zeros(nc); ds = torch.zeros(nc)
    for batch in loader:
        imgs, masks = batch["image"].to(device), batch["mask"].to(device)
        logits = model(imgs)
        total_loss += criterion(logits, masks).item() * imgs.size(0)
        preds = logits.argmax(1).cpu(); mc = masks.cpu()
        for c in range(nc):
            pc, tc = (preds==c), (mc==c)
            inter[c]+=(pc&tc).sum().float(); uni[c]+=(pc|tc).sum().float()
            di[c]+=(pc.float()*tc.float()).sum(); ds[c]+=pc.float().sum()+tc.float().sum()
        del logits, preds, mc
    piou = [(inter[c]/uni[c]).item() if uni[c]>0 else float("nan") for c in range(nc)]
    pdice = [(2*di[c]/ds[c]).item() if ds[c]>0 else float("nan") for c in range(nc)]
    vi=[x for x in piou if x==x]; vd=[x for x in pdice if x==x]
    return {"loss":total_loss/len(loader.dataset), "per_class_iou":piou,
            "mean_iou":sum(vi)/len(vi) if vi else 0, "per_class_dice":pdice,
            "mean_dice":sum(vd)/len(vd) if vd else 0}

def run_training(model, epochs, lr, wd, tag):
    criterion = DiceCELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    train_loader = DataLoader(train_set, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

    best = 0.0
    hist = {"train_loss":[],"val_loss":[],"val_miou":[],"val_dice":[],"lr":[]}
    print(f"\n{'='*90}")
    print(f"Training {tag} for {epochs} epochs...")
    print("-"*90)

    for epoch in range(1, epochs+1):
        t0 = time.time()
        tl = train_one_epoch(model, train_loader, criterion, optimizer)
        val = evaluate(model, val_loader, criterion, NC)
        scheduler.step()
        lr_now = optimizer.param_groups[0]["lr"]
        dt = time.time()-t0

        hist["train_loss"].append(tl); hist["val_loss"].append(val["loss"])
        hist["val_miou"].append(val["mean_iou"]); hist["val_dice"].append(val["mean_dice"])
        hist["lr"].append(lr_now)

        iou_str = " | ".join(f"{CLASS_NAMES[i]}: {v:.3f}" if v==v else f"{CLASS_NAMES[i]}: N/A"
                             for i,v in enumerate(val["per_class_iou"]))
        print(f"Epoch {epoch:3d}/{epochs} | train: {tl:.4f} | val: {val['loss']:.4f} | "
              f"mIoU: {val['mean_iou']:.4f} | Dice: {val['mean_dice']:.4f} | lr: {lr_now:.2e} | {dt:.0f}s")
        print(f"         {iou_str}")

        if val["mean_iou"] > best:
            best = val["mean_iou"]
            torch.save({"epoch":epoch, "model_state_dict":model.state_dict(),
                        "best_metric":best, "tag":tag}, f"best_{tag}.pt")
            print(f"         ** New best mIoU: {best:.4f} -- saved **")

        gc.collect(); torch.cuda.empty_cache()

    print(f"\n{tag} done. Best mIoU: {best:.4f}")
    return hist, best

## Run 1: ResNet-34 Baseline

In [ ]:
model_baseline = smp.Unet(
    encoder_name="resnet34", encoder_weights="imagenet",
    in_channels=3, classes=NC,
).to(device)
print(f"ResNet-34: {sum(p.numel() for p in model_baseline.parameters() if p.requires_grad):,} params")

hist_baseline, best_baseline = run_training(
    model_baseline, epochs=50, lr=1e-4, wd=1e-4, tag="resnet34"
)

## Run 2: DINOv2 Enhanced

In [ ]:
# Free baseline model from GPU
del model_baseline
gc.collect(); torch.cuda.empty_cache()

class DINOv2UNet(nn.Module):
    def __init__(self, classes=4, dropout_p=0.1):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
        self.embed_dim = self.backbone.embed_dim  # 768
        self.patch_size = 14
        n = len(self.backbone.blocks)
        self.feat_idx = [n//4-1, n//2-1, 3*n//4-1, n-1]

        ch = [64, 128, 256, 512]
        self.projections = nn.ModuleList([
            nn.Sequential(nn.Conv2d(self.embed_dim, c, 1), nn.BatchNorm2d(c), nn.ReLU(True))
            for c in ch])

        dec = [256, 128, 64, 32]
        self.decoders = nn.ModuleList()
        for i, dc in enumerate(dec):
            ic = ch[-(i+1)] + (dec[i-1] if i>0 else 0)
            self.decoders.append(nn.Sequential(
                nn.Conv2d(ic, dc, 3, padding=1), nn.BatchNorm2d(dc), nn.ReLU(True),
                nn.Dropout2d(dropout_p),
                nn.Conv2d(dc, dc, 3, padding=1), nn.BatchNorm2d(dc), nn.ReLU(True)))
        self.final = nn.Conv2d(dec[-1], classes, 1)

    def _extract(self, x):
        B,_,H,W = x.shape; h,w = H//self.patch_size, W//self.patch_size
        feats, hooks = [], []
        for idx in self.feat_idx:
            fl = []; feats.append(fl)
            hooks.append(self.backbone.blocks[idx].register_forward_hook(lambda m,i,o,s=fl: s.append(o)))
        self.backbone.forward_features(x)
        for hook in hooks: hook.remove()
        outs = []
        for i,(fl,proj) in enumerate(zip(feats, self.projections)):
            f = proj(fl[0][:,1:,:].reshape(B,h,w,self.embed_dim).permute(0,3,1,2))
            if i<3: f = F.interpolate(f, scale_factor=1.0/(2**(3-i)), mode="bilinear", align_corners=False)
            outs.append(f)
        return outs

    def forward(self, x):
        _,_,H,W = x.shape; enc = self._extract(x)
        d = self.decoders[0](enc[-1])
        for i in range(1, len(self.decoders)):
            d = F.interpolate(d, size=enc[-(i+1)].shape[2:], mode="bilinear", align_corners=False)
            d = self.decoders[i](torch.cat([d, enc[-(i+1)]], dim=1))
        return self.final(F.interpolate(d, size=(H,W), mode="bilinear", align_corners=False))

# DINOv2 needs input divisible by 14 -- use 476
train_set_dino = LunarDataset([all_imgs[i] for i in train_idx], [all_masks[i] for i in train_idx], get_transforms(476, True))
val_set_dino = LunarDataset([all_imgs[i] for i in val_idx], [all_masks[i] for i in val_idx], get_transforms(476, False))

# Temporarily swap datasets for DINOv2 training
train_set_orig, val_set_orig = train_set, val_set
train_set, val_set = train_set_dino, val_set_dino

model_dino = DINOv2UNet(classes=NC, dropout_p=0.1).to(device)
print(f"DINOv2-Base U-Net: {sum(p.numel() for p in model_dino.parameters() if p.requires_grad):,} params")

hist_dino, best_dino = run_training(
    model_dino, epochs=50, lr=5e-5, wd=1e-2, tag="dinov2"
)

# Restore original datasets
train_set, val_set = train_set_orig, val_set_orig

## Compare Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep1 = range(1, len(hist_baseline["val_miou"])+1)
ep2 = range(1, len(hist_dino["val_miou"])+1)

axes[0].plot(ep1, hist_baseline["val_miou"], label="ResNet-34", color="blue")
axes[0].plot(ep2, hist_dino["val_miou"], label="DINOv2", color="red")
axes[0].set(xlabel="Epoch", ylabel="mIoU", title="Validation mIoU")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep1, hist_baseline["val_dice"], label="ResNet-34", color="blue")
axes[1].plot(ep2, hist_dino["val_dice"], label="DINOv2", color="red")
axes[1].set(xlabel="Epoch", ylabel="Dice", title="Validation Dice")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle(f"ResNet-34 (best mIoU: {best_baseline:.4f}) vs DINOv2 (best mIoU: {best_dino:.4f})", fontsize=14)
plt.tight_layout(); plt.show()

## Test Evaluation (Best Model)

In [ ]:
# Pick the winner
if best_dino >= best_baseline:
    best_tag = "dinov2"
    best_model = model_dino
    test_set_final = LunarDataset([all_imgs[i] for i in test_idx], [all_masks[i] for i in test_idx], get_transforms(476, False))
else:
    best_tag = "resnet34"
    best_model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=NC).to(device)
    test_set_final = test_set

ckpt = torch.load(f"best_{best_tag}.pt", map_location=device, weights_only=False)
best_model.load_state_dict(ckpt["model_state_dict"])
print(f"Winner: {best_tag} (epoch {ckpt['epoch']}, mIoU: {ckpt['best_metric']:.4f})")

test_loader = DataLoader(test_set_final, batch_size=8, shuffle=False, num_workers=0, pin_memory=False)
criterion = DiceCELoss()
test = evaluate(best_model, test_loader, criterion, NC)

print(f"\nTest Loss: {test['loss']:.4f}  |  mIoU: {test['mean_iou']:.4f}  |  Dice: {test['mean_dice']:.4f}")
print("\nPer-class:")
for i in range(NC):
    iv = f"{test['per_class_iou'][i]:.4f}" if test['per_class_iou'][i]==test['per_class_iou'][i] else "N/A"
    dv = f"{test['per_class_dice'][i]:.4f}" if test['per_class_dice'][i]==test['per_class_dice'][i] else "N/A"
    print(f"  {CLASS_NAMES[i]:15s}: IoU={iv}  Dice={dv}")

## Predictions + MC Dropout Uncertainty

In [ ]:
@torch.no_grad()
def predict_one(model, img_t):
    model.eval()
    return model(img_t.unsqueeze(0).to(device)).argmax(1).squeeze(0).cpu().numpy()

def mc_predict(model, img_t, n_samples=20):
    model.eval()
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)): m.train()
    x = img_t.unsqueeze(0).to(device)
    probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            probs.append(F.softmax(model(x), dim=1).cpu())
    probs = torch.stack(probs).squeeze(1)
    mean_p = probs.mean(0)
    pred = mean_p.argmax(0).numpy()
    entropy = -(mean_p * torch.log(mean_p + 1e-10)).sum(0).numpy()
    return pred, entropy

has_dropout = best_tag == "dinov2"
ncols = 4 if has_dropout else 3
fig, axes = plt.subplots(4, ncols, figsize=(5*ncols, 20))
titles = ["Input", "Ground Truth", "Prediction"] + (["Uncertainty"] if has_dropout else [])
for c, t in enumerate(titles): axes[0,c].set_title(t, fontsize=14, fontweight="bold")

for row in range(4):
    s = test_set_final[row*50]
    img = s["image"]; gt = s["mask"].numpy()
    if has_dropout:
        pred, unc = mc_predict(best_model, img)
    else:
        pred = predict_one(best_model, img); unc = None
    axes[row,0].imshow(img.permute(1,2,0).numpy())
    axes[row,1].imshow(CLASS_COLORS[gt])
    axes[row,2].imshow(CLASS_COLORS[pred])
    if has_dropout: axes[row,3].imshow(unc, cmap="hot")

for ax in axes.flat: ax.axis("off")
from matplotlib.patches import Patch
legend = [Patch(facecolor=c/255, label=n) for c,n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=legend, loc="lower center", ncol=4, fontsize=12, bbox_to_anchor=(0.5,-0.02))
plt.suptitle(f"Test Predictions ({best_tag})", fontsize=16); plt.tight_layout(); plt.show()

## Sim-to-Real: Real Moon Images

In [ ]:
real_images = sorted(real_moon_dir.glob("*.png"))
isz = 476 if best_tag == "dinov2" else 480
fig, axes = plt.subplots(4, 3 if has_dropout else 2, figsize=(18, 20))
cols = ["Real Image", "Prediction"] + (["Uncertainty"] if has_dropout else [])
for c,t in enumerate(cols): axes[0,c].set_title(t, fontsize=14, fontweight="bold")

for row in range(4):
    img = cv2.cvtColor(cv2.imread(str(real_images[row*5])), cv2.COLOR_BGR2RGB)
    h,w = img.shape[:2]; cs = min(h,w,isz)
    cropped = A.CenterCrop(cs,cs)(image=img)["image"]
    if cs != isz: cropped = cv2.resize(cropped, (isz,isz))
    img_t = torch.from_numpy(cropped.transpose(2,0,1).astype(np.float32)/255.0)

    if has_dropout:
        pred, unc = mc_predict(best_model, img_t, n_samples=15)
    else:
        pred = predict_one(best_model, img_t); unc = None

    overlay = np.clip(0.6*cropped/255.0 + 0.4*CLASS_COLORS[pred]/255.0, 0, 1)
    axes[row,0].imshow(cropped); axes[row,0].set_ylabel(real_images[row*5].name, fontsize=9)
    axes[row,1].imshow(overlay)
    if has_dropout: axes[row,2].imshow(unc, cmap="hot")

for ax in axes.flat: ax.axis("off")
fig.legend(handles=legend, loc="lower center", ncol=4, fontsize=12, bbox_to_anchor=(0.5,-0.02))
plt.suptitle("Sim-to-Real Transfer", fontsize=16); plt.tight_layout(); plt.show()

## Save & Download

In [ ]:
results = {
    "best_model": best_tag,
    "baseline_best_miou": round(best_baseline, 4),
    "dinov2_best_miou": round(best_dino, 4),
    "best_epoch": ckpt["epoch"],
    "test_loss": round(test["loss"], 5),
    "test_mean_iou": round(test["mean_iou"], 4),
    "test_mean_dice": round(test["mean_dice"], 4),
    "test_per_class_iou": {CLASS_NAMES[i]: round(v,4) if v==v else None for i,v in enumerate(test["per_class_iou"])},
    "test_per_class_dice": {CLASS_NAMES[i]: round(v,4) if v==v else None for i,v in enumerate(test["per_class_dice"])},
    "baseline_history": hist_baseline,
    "dinov2_history": hist_dino,
}
with open("segmenter_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved. Winner: {best_tag}")
print(f"  ResNet-34 best mIoU: {best_baseline:.4f}")
print(f"  DINOv2    best mIoU: {best_dino:.4f}")
print(f"  Test mIoU: {test['mean_iou']:.4f}")
print(f"\nCopy to local project:")
print(f"  best_{best_tag}.pt -> models/best_segmenter.pt")

try:
    from google.colab import files
    files.download(f"best_{best_tag}.pt")
    files.download("segmenter_results.json")
    if best_tag != "resnet34":
        files.download("best_resnet34.pt")  # download both
except ImportError:
    print("Not in Colab")